# MedFactCheck — Storage Layer (MongoDB)
**Modulo:** Storage & Persistence  
**Database:** MongoDB (NoSQL)  
**Dipende da:** `ClaimsProcessing`, `RetrieverAgent`, `Reasoning_Veracity`  
**Espone a:** Dashboard Streamlit

---
### Architettura del modulo
```
ClaimsProcessing  ──→  save_claim_decomposition()   ──→  collezione: claims
RetrieverAgent    ──→  save_evidence()               ──→  collezione: evidence
Reasoning_Veracity ──→  save_verdict()               ──→  collezione: verdicts
                         aggregate_final_verdict()   ──→  collezione: final_results
Dashboard         ←──  get_*() / query_*() methods
```
### Collezioni MongoDB
| Collezione | Contenuto | Indici |
|---|---|---|
| `claims` | claim originale + sub-claims + routes | `claim_id`, `timestamp` |
| `evidence` | passage da PubMed/SciFact/ePMC per ogni sub-claim | `claim_id`, `source` |
| `verdicts` | CoT + label + confidence per ogni sub-claim | `claim_id`, `verdict` |
| `final_results` | verdetto aggregato + agent trace | `claim_id`, `final_verdict` |


## 1. Installazione dipendenze

In [ ]:
!pip install -q pymongo==4.10.1 python-dotenv==1.0.1
print("✅ Dipendenze storage installate.")

## 2. Configurazione connessione MongoDB

**Opzione A — MongoDB Atlas (cloud, gratuito con M0 cluster):**  
`MONGO_URI = "mongodb+srv://<user>:<password>@cluster0.xxxxx.mongodb.net/"`

**Opzione B — MongoDB locale:**  
`MONGO_URI = "mongodb://localhost:27017/"`

Inserisci la tua URI nella cella sottostante o salvala in un file `.env`.


In [ ]:
import os
from pymongo import MongoClient, ASCENDING, DESCENDING
from pymongo.errors import ConnectionFailure

# ── URI di connessione ────────────────────────────────────────────────────
# Opzione 1: da variabile d'ambiente / Colab Secrets
try:
    from google.colab import userdata
    MONGO_URI = userdata.get("MONGO_URI")
    print("✅ URI caricata da Colab Secrets.")
except Exception:
    # Opzione 2: inserisci direttamente qui
    MONGO_URI = "mongodb://localhost:27017/"  # ← sostituisci con Atlas URI se necessario
    print(f"⚠️  URI impostata manualmente: {MONGO_URI}")

DB_NAME = "medfactcheck"

# ── Test connessione ──────────────────────────────────────────────────────
try:
    _test_client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=5000)
    _test_client.admin.command("ping")
    print(f"✅ Connessione MongoDB OK → database: '{DB_NAME}'")
    _test_client.close()
except ConnectionFailure as e:
    print(f"❌ Connessione fallita: {e}")
    print("   Verifica che MongoDB sia in esecuzione o che l'URI Atlas sia corretta.")


## 3. Classe `StorageManager`

Interfaccia unica per tutte le operazioni di lettura/scrittura.  
Tutti gli altri notebook importano **solo questa classe**.


In [ ]:
import hashlib
import uuid
from datetime import datetime, timezone
from typing import Optional

from pymongo import MongoClient, ASCENDING, DESCENDING
from pymongo.collection import Collection


class StorageManager:
    """
    Gestisce la persistenza di tutti i dati di MedFactCheck su MongoDB.

    Collezioni:
      - claims         : claim originale + sub-claims con routes (da ClaimsProcessing)
      - evidence       : passage recuperati per ogni sub-claim (da RetrieverAgent)
      - verdicts       : CoT + label + confidence per ogni sub-claim (da Reasoning_Veracity)
      - final_results  : verdetto aggregato + agent trace (per la Dashboard)
    """

    # ── Inizializzazione ──────────────────────────────────────────────────────

    def __init__(self, mongo_uri: str = MONGO_URI, db_name: str = DB_NAME):
        self._client = MongoClient(mongo_uri, serverSelectionTimeoutMS=5000)
        self._db     = self._client[db_name]

        self.claims        : Collection = self._db["claims"]
        self.evidence      : Collection = self._db["evidence"]
        self.verdicts      : Collection = self._db["verdicts"]
        self.final_results : Collection = self._db["final_results"]

        self._ensure_indexes()
        print(f"✅ StorageManager inizializzato → DB: '{db_name}'")

    def _ensure_indexes(self):
        """Crea gli indici necessari (idempotente)."""
        # claims
        self.claims.create_index([("claim_id", ASCENDING)], unique=True)
        self.claims.create_index([("timestamp", DESCENDING)])

        # evidence
        self.evidence.create_index([("claim_id", ASCENDING)])
        self.evidence.create_index([("sub_claim_hash", ASCENDING)])
        self.evidence.create_index([("source", ASCENDING)])

        # verdicts
        self.verdicts.create_index([("claim_id", ASCENDING)])
        self.verdicts.create_index([("verdict", ASCENDING)])
        self.verdicts.create_index([("timestamp", DESCENDING)])

        # final_results
        self.final_results.create_index([("claim_id", ASCENDING)], unique=True)
        self.final_results.create_index([("final_verdict", ASCENDING)])
        self.final_results.create_index([("timestamp", DESCENDING)])
        print("   📌 Indici verificati/creati.")

    def close(self):
        self._client.close()

    # ── Utility interna ───────────────────────────────────────────────────────

    @staticmethod
    def _make_claim_id(text: str) -> str:
        """Genera un claim_id deterministico dall'hash del testo."""
        digest = hashlib.md5(text.lower().strip().encode()).hexdigest()[:10]
        return f"claim_{digest}"

    @staticmethod
    def _now() -> str:
        return datetime.now(timezone.utc).isoformat()

    # ═══════════════════════════════════════════════════════════════════════════
    # COLLEZIONE: claims
    # Input: output di ClaimsProcessing (QwenNF4Decomposer.decompose())
    # ═══════════════════════════════════════════════════════════════════════════

    def save_claim_decomposition(
        self,
        original_text: str,
        decomposer_output: dict,
        source_type: str = "text"
    ) -> str:
        """
        Salva il claim originale e i suoi sub-claims con routing.

        Args:
            original_text    : testo grezzo inviato dal Decomposer
            decomposer_output: dizionario restituito da QwenNF4Decomposer.decompose()
                               Struttura attesa:
                               {
                                 "reasoning": "...",
                                 "sub_claims": [
                                   {"claim": "...", "routes": ["kb"]},
                                   {"claim": "...", "routes": ["kb", "lit"]}
                                 ]
                               }
            source_type      : "text" | "url" | "image"

        Returns:
            claim_id (str)
        """
        claim_id  = self._make_claim_id(original_text)
        timestamp = self._now()

        # Arricchiamo ogni sub-claim con un hash univoco e metadati
        sub_claims_enriched = []
        for sc in decomposer_output.get("sub_claims", []):
            sub_claims_enriched.append({
                "claim"         : sc["claim"],
                "routes"        : sc.get("routes", []),
                "sub_claim_hash": hashlib.md5(sc["claim"].encode()).hexdigest()[:8],
                "verdict_ready" : False   # diventa True dopo Reasoning_Veracity
            })

        document = {
            "claim_id"      : claim_id,
            "original_text" : original_text,
            "source_type"   : source_type,
            "decomposer_reasoning": decomposer_output.get("reasoning", ""),
            "sub_claims"    : sub_claims_enriched,
            "n_sub_claims"  : len(sub_claims_enriched),
            "timestamp"     : timestamp,
            "status"        : "decomposed"  # decomposed → retrieved → verified → done
        }

        self.claims.update_one(
            {"claim_id": claim_id},
            {"$set": document},
            upsert=True
        )
        print(f"💾 [claims] Salvato '{claim_id}' — {len(sub_claims_enriched)} sub-claim/s")
        return claim_id

    def get_claim(self, claim_id: str) -> Optional[dict]:
        """Recupera un claim completo per claim_id."""
        return self.claims.find_one({"claim_id": claim_id}, {"_id": 0})

    def update_claim_status(self, claim_id: str, status: str):
        """Aggiorna lo status del claim (decomposed→retrieved→verified→done)."""
        self.claims.update_one(
            {"claim_id": claim_id},
            {"$set": {"status": status, "last_updated": self._now()}}
        )

    # ═══════════════════════════════════════════════════════════════════════════
    # COLLEZIONE: evidence
    # Input: output di RetrieverAgent (retrieve_evidence())
    # ═══════════════════════════════════════════════════════════════════════════

    def save_evidence(self, claim_id: str, retriever_output: dict) -> list[str]:
        """
        Salva i passage di evidenza recuperati dal RetrieverAgent.

        Args:
            claim_id        : ID del claim padre (da save_claim_decomposition)
            retriever_output: dizionario restituito da retrieve_evidence()
                              Struttura attesa:
                              {
                                "claim_id": "...",
                                "claim"   : "testo del sub-claim",
                                "claim_analysis": {...},
                                "evidence_passages": [
                                  {
                                    "source": "PubMed|SciFact|EuropePMC",
                                    "doc_id": "...", "pmid": "...", "doi": "...",
                                    "title": "...", "abstract": "...", "text": "...",
                                    "authors": "...", "journal": "...", "year": "...",
                                    "rrf_score": 0.045,
                                    "relevance_score": 0.9,
                                    "verdict_direction": "supports|refutes|neutral",
                                    "evidence_type": "RCT|meta-analysis|...",
                                    "key_finding": "..."
                                  }, ...
                                ],
                                "stats": {...},
                                "errors": [...]
                              }

        Returns:
            lista di evidence_id inseriti
        """
        sub_claim_text = retriever_output.get("claim", "")
        sub_claim_hash = hashlib.md5(sub_claim_text.encode()).hexdigest()[:8]
        passages       = retriever_output.get("evidence_passages", [])
        evidence_ids   = []

        docs_to_insert = []
        for p in passages:
            evidence_id = f"ev_{claim_id}_{sub_claim_hash}_{p.get('doc_id','?')[:8]}"
            doc = {
                "evidence_id"       : evidence_id,
                "claim_id"          : claim_id,
                "sub_claim"         : sub_claim_text,
                "sub_claim_hash"    : sub_claim_hash,
                "claim_analysis"    : retriever_output.get("claim_analysis", {}),
                # — campi passage —
                "source"            : p.get("source", ""),
                "doc_id"            : p.get("doc_id", ""),
                "pmid"              : p.get("pmid", ""),
                "doi"               : p.get("doi", ""),
                "pmcid"             : p.get("pmcid", ""),
                "title"             : p.get("title", ""),
                "testo"             : p.get("text") or p.get("abstract") or p.get("testo", ""),
                "authors"           : p.get("authors", ""),
                "journal"           : p.get("journal", ""),
                "year"              : p.get("year", ""),
                "has_fulltext"      : p.get("has_fulltext", False),
                "url"               : p.get("url", ""),
                # — score di retrieval —
                "rrf_score"         : p.get("rrf_score", 0.0),
                "relevance_score"   : p.get("relevance_score", None),
                "verdict_direction" : p.get("verdict_direction", "neutral"),
                "evidence_type"     : p.get("evidence_type", "other"),
                "key_finding"       : p.get("key_finding", ""),
                # — metadati —
                "retrieval_stats"   : retriever_output.get("stats", {}),
                "retriever_errors"  : retriever_output.get("errors", []),
                "timestamp"         : self._now()
            }
            docs_to_insert.append(doc)
            evidence_ids.append(evidence_id)

        if docs_to_insert:
            for doc in docs_to_insert:
                self.evidence.update_one(
                    {"evidence_id": doc["evidence_id"]},
                    {"$set": doc},
                    upsert=True
                )
            print(f"💾 [evidence] {len(docs_to_insert)} passage salvati per '{claim_id}' (sub: {sub_claim_hash})")
        else:
            print(f"⚠️  [evidence] Nessun passage da salvare per '{claim_id}'")

        return evidence_ids

    def get_evidence_for_claim(self, claim_id: str) -> list[dict]:
        """Recupera tutti i passage di evidenza per un claim_id."""
        return list(self.evidence.find(
            {"claim_id": claim_id},
            {"_id": 0}
        ).sort("relevance_score", DESCENDING))

    def get_evidence_for_sub_claim(self, claim_id: str, sub_claim_hash: str) -> list[dict]:
        """Recupera i passage relativi a uno specifico sub-claim."""
        return list(self.evidence.find(
            {"claim_id": claim_id, "sub_claim_hash": sub_claim_hash},
            {"_id": 0}
        ).sort("rrf_score", DESCENDING))

    # ═══════════════════════════════════════════════════════════════════════════
    # COLLEZIONE: verdicts
    # Input: output di Reasoning_Veracity (ReasoningAndVeracityPipeline.process_claim())
    # ═══════════════════════════════════════════════════════════════════════════

    def save_verdict(self, claim_id: str, veracity_output: dict) -> str:
        """
        Salva il verdetto (CoT + label + confidence) di un sub-claim.

        Args:
            claim_id       : ID del claim padre
            veracity_output: dizionario restituito da process_claim()
                             Struttura attesa:
                             {
                               "claim"              : "testo del sub-claim",
                               "verdict"            : "Supported|Refuted|Not Enough Information",
                               "confidence_score"   : 0.91,
                               "chain_of_thought_log": "...",
                               "supporting_evidence": [
                                 {"titolo": "...", "url": "...", "testo": "..."}
                               ]
                             }

        Returns:
            verdict_id (str)
        """
        sub_claim_text = veracity_output.get("claim", "")
        sub_claim_hash = hashlib.md5(sub_claim_text.encode()).hexdigest()[:8]
        verdict_id     = f"vrd_{claim_id}_{sub_claim_hash}"

        document = {
            "verdict_id"          : verdict_id,
            "claim_id"            : claim_id,
            "sub_claim"           : sub_claim_text,
            "sub_claim_hash"      : sub_claim_hash,
            "verdict"             : veracity_output.get("verdict", "Not Enough Information"),
            "confidence_score"    : veracity_output.get("confidence_score", 0.0),
            "chain_of_thought_log": veracity_output.get("chain_of_thought_log", ""),
            "supporting_evidence" : veracity_output.get("supporting_evidence", []),
            "timestamp"           : self._now()
        }

        self.verdicts.update_one(
            {"verdict_id": verdict_id},
            {"$set": document},
            upsert=True
        )

        # Aggiorna il flag verdict_ready nel documento claims
        self.claims.update_one(
            {"claim_id": claim_id, "sub_claims.sub_claim_hash": sub_claim_hash},
            {"$set": {"sub_claims.$.verdict_ready": True}}
        )

        print(f"💾 [verdicts] '{verdict_id}' → {document['verdict']} ({document['confidence_score']:.2%})")
        return verdict_id

    def get_verdicts_for_claim(self, claim_id: str) -> list[dict]:
        """Recupera tutti i verdetti per un claim_id."""
        return list(self.verdicts.find(
            {"claim_id": claim_id},
            {"_id": 0}
        ).sort("timestamp", ASCENDING))

    # ═══════════════════════════════════════════════════════════════════════════
    # COLLEZIONE: final_results
    # Aggregazione per la Dashboard — chiamata dal Coordinator Agent
    # ═══════════════════════════════════════════════════════════════════════════

    def aggregate_final_verdict(self, claim_id: str, agent_trace: list = None) -> dict:
        """
        Aggrega i verdetti dei sub-claim in un verdetto finale e lo salva.

        Logica di aggregazione:
          - Almeno 1 sub-claim Refuted  → finale = Refuted
          - Tutti Supported             → finale = Supported
          - Altrimenti                  → finale = Not Enough Information

        Args:
            claim_id   : ID del claim
            agent_trace: lista opzionale di step dell'agent (per la dashboard)

        Returns:
            documento final_result completo
        """
        claim_doc    = self.get_claim(claim_id)
        sub_verdicts = self.get_verdicts_for_claim(claim_id)

        if not sub_verdicts:
            print(f"⚠️  [final_results] Nessun verdetto trovato per '{claim_id}'")
            return {}

        labels      = [v["verdict"] for v in sub_verdicts]
        avg_conf    = round(sum(v["confidence_score"] for v in sub_verdicts) / len(sub_verdicts), 4)

        # Regola di aggregazione
        if "Refuted" in labels:
            final_verdict = "Refuted"
        elif all(l == "Supported" for l in labels):
            final_verdict = "Supported"
        else:
            final_verdict = "Not Enough Information"

        document = {
            "claim_id"          : claim_id,
            "original_text"     : claim_doc.get("original_text", "") if claim_doc else "",
            "final_verdict"     : final_verdict,
            "avg_confidence"    : avg_conf,
            "n_sub_claims"      : len(sub_verdicts),
            "verdict_breakdown" : {
                "Supported"             : labels.count("Supported"),
                "Refuted"               : labels.count("Refuted"),
                "Not Enough Information": labels.count("Not Enough Information")
            },
            "sub_verdicts"      : sub_verdicts,
            "agent_trace"       : agent_trace or [],
            "timestamp"         : self._now()
        }

        self.final_results.update_one(
            {"claim_id": claim_id},
            {"$set": document},
            upsert=True
        )

        # Aggiorna status nel documento claims
        self.update_claim_status(claim_id, "done")

        print(f"🏁 [final_results] '{claim_id}' → VERDETTO FINALE: {final_verdict} (conf media: {avg_conf:.2%})")
        return document

    # ═══════════════════════════════════════════════════════════════════════════
    # METODI DI QUERY — per la Dashboard Streamlit
    # ═══════════════════════════════════════════════════════════════════════════

    def get_final_result(self, claim_id: str) -> Optional[dict]:
        """Recupera il risultato finale completo per claim_id."""
        return self.final_results.find_one({"claim_id": claim_id}, {"_id": 0})

    def get_all_results(
        self,
        verdict_filter: str = None,
        limit: int = 50,
        skip: int = 0
    ) -> list[dict]:
        """
        Recupera i risultati finali per la dashboard (con filtri opzionali).

        Args:
            verdict_filter: "Supported" | "Refuted" | "Not Enough Information" | None
            limit         : numero massimo di risultati
            skip          : offset per paginazione
        """
        query = {}
        if verdict_filter:
            query["final_verdict"] = verdict_filter

        return list(
            self.final_results
            .find(query, {"_id": 0})
            .sort("timestamp", DESCENDING)
            .skip(skip)
            .limit(limit)
        )

    def search_claims(self, keyword: str, limit: int = 20) -> list[dict]:
        """Ricerca full-text nel testo originale dei claim."""
        self.final_results.create_index([("original_text", "text")], background=True)
        return list(
            self.final_results
            .find(
                {"$text": {"$search": keyword}},
                {"_id": 0, "score": {"$meta": "textScore"}}
            )
            .sort([("score", {"$meta": "textScore"})])
            .limit(limit)
        )

    def get_stats(self) -> dict:
        """Statistiche globali per la dashboard."""
        total  = self.final_results.count_documents({})
        if total == 0:
            return {"total": 0, "supported": 0, "refuted": 0, "nei": 0, "avg_confidence": 0.0}

        pipeline = [
            {"$group": {
                "_id"           : "$final_verdict",
                "count"         : {"$sum": 1},
                "avg_confidence": {"$avg": "$avg_confidence"}
            }}
        ]
        agg = {r["_id"]: r for r in self.final_results.aggregate(pipeline)}

        return {
            "total"          : total,
            "supported"      : agg.get("Supported",              {}).get("count", 0),
            "refuted"        : agg.get("Refuted",                {}).get("count", 0),
            "nei"            : agg.get("Not Enough Information", {}).get("count", 0),
            "avg_confidence" : round(
                sum(r.get("avg_confidence", 0.0) for r in agg.values()) / len(agg), 4
            ) if agg else 0.0,
            "sources_used"   : list(self.evidence.distinct("source"))
        }

    def delete_claim(self, claim_id: str) -> dict:
        """Elimina un claim e tutti i dati associati (cascade delete)."""
        r1 = self.claims.delete_many({"claim_id": claim_id})
        r2 = self.evidence.delete_many({"claim_id": claim_id})
        r3 = self.verdicts.delete_many({"claim_id": claim_id})
        r4 = self.final_results.delete_many({"claim_id": claim_id})
        summary = {
            "claims_deleted"  : r1.deleted_count,
            "evidence_deleted": r2.deleted_count,
            "verdicts_deleted": r3.deleted_count,
            "results_deleted" : r4.deleted_count
        }
        print(f"🗑️  Eliminati tutti i dati per '{claim_id}': {summary}")
        return summary


print("✅ Classe StorageManager definita.")


## 4. Istanziazione e verifica

In [ ]:
# Istanza globale (usata in tutto il notebook e dagli altri moduli)
storage = StorageManager(mongo_uri=MONGO_URI, db_name=DB_NAME)

# Verifica stats iniziali
stats = storage.get_stats()
print(f"\n📊 Stato attuale del DB:")
print(f"   Claim verificati : {stats['total']}")
print(f"   Supported        : {stats['supported']}")
print(f"   Refuted          : {stats['refuted']}")
print(f"   Not Enough Info  : {stats['nei']}")
print(f"   Confidence media : {stats['avg_confidence']:.2%}")


## 5. Integrazione con `ClaimsProcessing`

Copia questo snippet alla **fine** del notebook `ClaimsProcessing`, dopo che `decomposer.decompose()` restituisce il JSON.


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  SNIPPET DA INCOLLARE IN ClaimsProcessing.ipynb                     ║
# ║  Posizione: subito dopo decomposer.decompose() restituisce l'output ║
# ╚══════════════════════════════════════════════════════════════════════╝

# Simuliamo l'output reale del Decomposer
mock_decomposer_output = {
    "reasoning": "Classificazione farmacologica (kb) e azione clinica (kb, lit).",
    "sub_claims": [
        {
            "claim" : "L'ibuprofene è un antinfiammatorio non steroideo.",
            "routes": ["kb"]
        },
        {
            "claim" : "L'ibuprofene riduce drasticamente il dolore articolare.",
            "routes": ["kb", "lit"]
        }
    ]
}
original_input = "L'ibuprofene è un antinfiammatorio non steroideo. Riduce drasticamente il dolore articolare."

# ── SALVATAGGIO (1 riga) ─────────────────────────────────────────────────────
claim_id = storage.save_claim_decomposition(
    original_text    = original_input,
    decomposer_output= mock_decomposer_output,
    source_type      = "text"   # oppure "url" o "image"
)

print(f"\n✅ claim_id da passare al RetrieverAgent: '{claim_id}'")

# Verifica lettura
saved = storage.get_claim(claim_id)
print(f"\n📄 Claim salvato:")
print(f"   Original : {saved['original_text'][:80]}")
print(f"   Status   : {saved['status']}")
print(f"   Sub-claim: {saved['n_sub_claims']}")
for sc in saved['sub_claims']:
    print(f"     • [{','.join(sc['routes'])}] {sc['claim']}")


## 6. Integrazione con `RetrieverAgent`

Copia questo snippet alla **fine** del notebook `RetrieverAgent`, dopo ogni chiamata a `retrieve_evidence()`.


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  SNIPPET DA INCOLLARE IN RetrieverAgent.ipynb                       ║
# ║  Posizione: dentro il loop su sub_claims, dopo retrieve_evidence()  ║
# ╚══════════════════════════════════════════════════════════════════════╝

# Simuliamo l'output reale di retrieve_evidence()
mock_retriever_output = {
    "claim_id"      : claim_id,   # <- viene dal ClaimsProcessing
    "claim"         : "L'ibuprofene riduce drasticamente il dolore articolare.",
    "claim_analysis": {
        "claim_type"  : "therapeutic",
        "key_concepts": ["ibuprofen", "pain relief", "articular pain"],
        "entities"    : {"drugs_or_interventions": ["ibuprofen"], "diseases": ["arthralgia"]}
    },
    "evidence_passages": [
        {
            "source"          : "PubMed",
            "doc_id"          : "12345678",
            "pmid"            : "12345678",
            "doi"             : "10.1016/example",
            "title"           : "Ibuprofen efficacy in articular pain: a meta-analysis",
            "abstract"        : "Meta-analysis di 12 RCT mostra riduzione significativa del dolore.",
            "text"            : "Meta-analysis di 12 RCT mostra riduzione significativa del dolore.",
            "authors"         : "Rossi M, Bianchi A et al.",
            "journal"         : "Pain",
            "year"            : "2021",
            "has_fulltext"    : False,
            "url"             : "https://pubmed.ncbi.nlm.nih.gov/12345678/",
            "rrf_score"       : 0.0423,
            "relevance_score" : 0.91,
            "verdict_direction": "supports",
            "evidence_type"   : "meta-analysis",
            "key_finding"     : "Ibuprofen significantly reduces articular pain scores."
        },
        {
            "source"          : "SciFact",
            "doc_id"          : "scifact_789",
            "pmid"            : "",
            "doi"             : "",
            "title"           : "NSAID mechanisms and clinical use",
            "abstract"        : "Review sui meccanismi degli NSAID incluso ibuprofene.",
            "text"            : "Review sui meccanismi degli NSAID incluso ibuprofene.",
            "authors"         : "Smith J",
            "journal"         : "Pharmacology Reviews",
            "year"            : "2020",
            "has_fulltext"    : False,
            "url"             : "",
            "rrf_score"       : 0.0311,
            "relevance_score" : 0.78,
            "verdict_direction": "supports",
            "evidence_type"   : "review",
            "key_finding"     : "NSAIDs inhibit COX enzymes reducing prostaglandin synthesis."
        }
    ],
    "stats"  : {"pubmed_count": 1, "scifact_count": 1, "total_passages": 2},
    "errors" : []
}

# ── SALVATAGGIO (1 riga) ─────────────────────────────────────────────────────
evidence_ids = storage.save_evidence(
    claim_id        = claim_id,
    retriever_output= mock_retriever_output
)

# Aggiorna lo status del claim
storage.update_claim_status(claim_id, "retrieved")

print(f"\n✅ {len(evidence_ids)} passage salvati: {evidence_ids}")

# Lettura per verifica (o per passare al Reasoning Agent)
passages = storage.get_evidence_for_claim(claim_id)
print(f"\n📚 Evidenze disponibili per il Reasoning Agent:")
for p in passages:
    print(f"   [{p['source']}] {p['title'][:60]} | score: {p['relevance_score']}")


## 7. Integrazione con `Reasoning_Veracity`

Copia questo snippet alla **fine** del notebook `Reasoning_Veracity`, dopo ogni chiamata a `pipeline.process_claim()`.


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  SNIPPET DA INCOLLARE IN Reasoning_Veracity.ipynb                   ║
# ║  Posizione: dopo my_pipeline.process_claim() per ogni sub-claim     ║
# ╚══════════════════════════════════════════════════════════════════════╝

# Simuliamo l'output reale di process_claim()
mock_veracity_output = {
    "claim"               : "L'ibuprofene riduce drasticamente il dolore articolare.",
    "verdict"             : "Supported",
    "confidence_score"    : 0.9134,
    "chain_of_thought_log": (
        "Il claim afferma una riduzione del dolore articolare da parte dell'ibuprofene. "
        "Le evidenze mostrano una meta-analisi di 12 RCT con risultati statisticamente "
        "significativi. I meccanismi COX-inibitori degli NSAID spiegano l'effetto. "
        "Allineamento semantico quasi totale tra claim ed evidenze."
    ),
    "supporting_evidence" : [
        {
            "titolo": "Ibuprofen efficacy in articular pain: a meta-analysis",
            "url"   : "https://pubmed.ncbi.nlm.nih.gov/12345678/",
            "testo" : "Meta-analysis di 12 RCT mostra riduzione significativa del dolore."
        }
    ]
}

# ── SALVATAGGIO (1 riga) ─────────────────────────────────────────────────────
verdict_id = storage.save_verdict(
    claim_id       = claim_id,
    veracity_output= mock_veracity_output
)

storage.update_claim_status(claim_id, "verified")
print(f"\n✅ Verdetto salvato: '{verdict_id}'")

# Recupera per la dashboard
v = storage.get_verdicts_for_claim(claim_id)[0]
print(f"\n🔬 Sub-claim  : {v['sub_claim'][:70]}")
print(f"   Verdetto   : {v['verdict']}")
print(f"   Confidence : {v['confidence_score']:.2%}")
print(f"   CoT (preview): {v['chain_of_thought_log'][:120]}...")


## 8. Aggregazione verdetto finale (Coordinator Agent)

Chiamata **una sola volta** dopo che **tutti** i sub-claim hanno il loro verdetto.  
Produce il documento in `final_results` che la Dashboard legge direttamente.


In [ ]:
# Salviamo anche il secondo sub-claim (KB-only, in genere non passa per Veracity)
# ma lo aggiungiamo per completezza del test

storage.save_verdict(claim_id, {
    "claim"               : "L'ibuprofene è un antinfiammatorio non steroideo.",
    "verdict"             : "Supported",
    "confidence_score"    : 0.9812,
    "chain_of_thought_log": "Classificazione farmacologica confermata dalla KB.",
    "supporting_evidence" : []
})

# ── Agent trace (opzionale, registra la sequenza degli agenti) ───────────────
agent_trace = [
    {"agent": "Decomposer",        "model": "Qwen2.5-VL-7B", "status": "ok", "n_sub_claims": 2},
    {"agent": "RetrieverAgent",    "model": "BioBERT+BM25",  "status": "ok", "n_passages": 2},
    {"agent": "ReasoningVeracity", "model": "Qwen2.5-7B + DeBERTa-v3", "status": "ok"}
]

# ── AGGREGAZIONE FINALE (1 riga) ─────────────────────────────────────────────
final = storage.aggregate_final_verdict(claim_id, agent_trace=agent_trace)

print(f"\n{'='*60}")
print(f"  VERDETTO FINALE PER IL CLAIM")
print(f"{'='*60}")
print(f"  Testo    : {final['original_text'][:80]}")
print(f"  Verdetto : {final['final_verdict']}")
print(f"  Conf.    : {final['avg_confidence']:.2%}")
print(f"  Breakdown: {final['verdict_breakdown']}")


## 9. API di query per la Dashboard Streamlit

Questi sono tutti i metodi che la dashboard dovrà chiamare sullo `StorageManager`.


In [ ]:
# ── 1. Tutti i claim verificati (con filtro opzionale) ────────────────────
print("\n📋 Ultimi claim verificati:")
results = storage.get_all_results(limit=10)
for r in results:
    print(f"   [{r['final_verdict']:25s}] {r['original_text'][:60]}")

# ── 2. Filtra solo i Refuted ──────────────────────────────────────────────
print("\n🔴 Claim Refuted:")
refuted = storage.get_all_results(verdict_filter="Refuted")
print(f"   Trovati: {len(refuted)}")

# ── 3. Ricerca per keyword ────────────────────────────────────────────────
print("\n🔍 Ricerca 'ibuprofene':")
found = storage.search_claims("ibuprofene")
for r in found:
    print(f"   → {r['original_text'][:70]}")

# ── 4. Statistiche globali (per la header della dashboard) ────────────────
print("\n📊 Statistiche globali:")
stats = storage.get_stats()
for k, v in stats.items():
    print(f"   {k:<20}: {v}")

# ── 5. Dettaglio completo di un claim ─────────────────────────────────────
print("\n📑 Dettaglio claim:")
detail = storage.get_final_result(claim_id)
if detail:
    print(f"   ID       : {detail['claim_id']}")
    print(f"   Verdetto : {detail['final_verdict']}")
    print(f"   N. passi : {len(detail['agent_trace'])}")
    print(f"   Evidenze per sub-claim:")
    ev = storage.get_evidence_for_claim(claim_id)
    for e in ev:
        print(f"     [{e['source']}] {e['title'][:55]} — dir: {e['verdict_direction']}")


## 10. Snippet per la Dashboard Streamlit

Blocco pronto da incollare all'inizio di `app.py` (o `dashboard.py`).


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  SNIPPET DA INCOLLARE IN dashboard/app.py (Streamlit)               ║
# ╚══════════════════════════════════════════════════════════════════════╝

STREAMLIT_SNIPPET = '''
import streamlit as st
from storage_manager import StorageManager   # ← esporta la classe in un file .py

@st.cache_resource
def get_storage():
    return StorageManager()

storage = get_storage()

# ── Sidebar: statistiche e filtri ────────────────────────────────────────
with st.sidebar:
    stats = storage.get_stats()
    st.metric("Claim Verificati",  stats["total"])
    st.metric("✅ Supported",      stats["supported"])
    st.metric("❌ Refuted",        stats["refuted"])
    st.metric("❓ NEI",            stats["nei"])
    st.metric("Conf. Media",       f"{stats['avg_confidence']:.1%}")

    verdict_filter = st.selectbox(
        "Filtra per verdetto",
        ["Tutti", "Supported", "Refuted", "Not Enough Information"]
    )
    keyword = st.text_input("🔍 Cerca per keyword")
    threshold = st.slider("Soglia confidence", 0.0, 1.0, 0.5)

# ── Main: lista claim ─────────────────────────────────────────────────────
vf = None if verdict_filter == "Tutti" else verdict_filter
if keyword:
    results = storage.search_claims(keyword)
else:
    results = storage.get_all_results(verdict_filter=vf)

results = [r for r in results if r.get("avg_confidence", 0) >= threshold]

for r in results:
    color = {"Supported": "🟢", "Refuted": "🔴"}.get(r["final_verdict"], "🟡")
    with st.expander(f"{color} {r['original_text'][:80]}"):
        st.write(f"**Verdetto:** {r['final_verdict']}  |  **Confidence:** {r['avg_confidence']:.1%}")
        for sv in r.get("sub_verdicts", []):
            st.markdown(f"- `{sv['verdict']}` — {sv['sub_claim'][:70]}")
            st.caption(sv["chain_of_thought_log"][:200])
        # Agent trace
        with st.expander("🔧 Agent trace"):
            st.json(r.get("agent_trace", []))
'''

print("📋 Snippet Streamlit pronto. Copia il contenuto di STREAMLIT_SNIPPET in dashboard/app.py")
print("   Ricorda di esportare StorageManager in un file 'storage_manager.py' separato.")


## 11. Cleanup dati di test (opzionale)

Rimuove i dati mock inseriti durante i test di questo notebook.


In [ ]:
# ── Decommenta per eliminare i dati di test ──────────────────────────────
# summary = storage.delete_claim(claim_id)
# print(f"Rimosso claim di test '{claim_id}':", summary)

print(f"ℹ️  Dato di test '{claim_id}' mantenuto nel DB.")
print("   Decommenta la riga sopra per eliminarlo.")
storage.close()
print("✅ Connessione MongoDB chiusa.")
